In [1]:
import os, sys, time, logging, importlib
import pandas as pd
sys.path.append(os.path.abspath(".."))
from utils import log

importlib.reload(log)

<module 'utils.log' from 'd:\\Work\\IRG\\hal_insight\\utils\\log.py'>

## test 11-19 (skip): fnege_pdf => fenege_table, one by one 

In [ ]:
#list de dicts
extraction_config=[
    {"input_path":'xxxxx',
    "year":2019,
     "start_page":13,
     "end_page":33,
     "output_path":"xxxxx"
    }
]

In [97]:
from pypdf import PdfReader, PdfWriter
import os
import camelot
import pandas as pd
year=2019#*
start_page, end_page=13, 33#**

output_folder='fnege/fnege_tables'
os.makedirs(output_folder, exist_ok=True)
input_pdf = f"fnege/fnege_pdf\Classement FNEGE {year}.pdf"

output_file = f"fnege_{year}.pdf"
output_path=os.path.join(output_folder, output_file)


reader = PdfReader(input_pdf)
writer = PdfWriter()
#read pdf
for page_num in range(start_page-1, end_page):
    writer.add_page(reader.pages[page_num])

#save
with open(output_path, "wb") as f:
    writer.write(f)
    logging.info(f"[SAVE] fnege pdf p{start_page}-{end_page} to {output_path}")
    


21:27 | [SAVE] fnege pdf p13-33 to fnege/fnege_tables\fnege_2019.pdf


In [98]:
os.path.basename(output_path)

'fnege_2019.pdf'

In [99]:

#read tables
tables = camelot.read_pdf(output_path, pages="all")#camelot只提取
print(len(tables))
df_list = [t.df for t in tables]
df = pd.concat(df_list, ignore_index=True)
display(df)

21


,0,1,2,3
0,@grh,2034-9130,3,HRM
1,Abacus,0001-3072,2,ACC
2,Academy of Management Annals,1941-6520,1,GEN MAN
3,Academy of Management Journal,0001-4273,1,GEN MAN
4,Academy of Management Learning \nand Education,1537-260X,2,GEN MAN
...,...,...,...,...
479,Venture Capital: An International \nJournal of...,1369-1066,4,FIN
480,Voluntas,0957-8765,4,GEN MAN
481,Work and Occupations,0730-8884,3,HRM
482,Work and Stress,0267-8373,3,HRM


In [ ]:
df_list = [t.df for t in tables]
df = pd.concat(df_list, ignore_index=True)
df.columns=['nom de la revue',"issn",'rang','domaine']
display(df)

In [ ]:
print(df['nom de la revue'].value_counts(dropna=False).sort_values(ascending=False))

In [ ]:
df.to_csv(f"fnege_tables/fnege_{year}.csv", index=False)

In [ ]:
desired_order = ['nom de la revue', 'issn', 'rang', 'domaine']
df = fnege2011[desired_order]

display(df)
df.to_csv(f"fnege_tables/fnege_2011.csv", index=False)

## *exception 22 (skip):

In [ ]:
import camelot
import pandas as pd

input_pdf = "fnege_pdf\FNEGE-classement-revues-de-gestion-2022.pdf"
# 坐标顺序： [x1, y1, x2, y2] （单位是 pt，PDF 坐标系左下原点）
# x1,y1 → 左上角; x2,y2 → 右下角
table_areas = ["30,750,650,30"]  

tables = camelot.read_pdf(
    input_pdf,
    pages="1-13",#all
    flavor="stream",  # 对复杂表格更稳健
    table_areas=table_areas
)
print("Tables détectées :", len(tables))

if len(tables)>0:
    df_list = [t.df for t in tables]
    df = pd.concat(df_list, ignore_index=True)

df.columns = ["issn", "essin", "nom de la revue", "domaine", "rang19", "rang"]

for i in range(1, len(df)):
    if pd.isna(df.loc[i, 'issn']) or df.loc[i, 'issn'] == "":
        # 将 title 合并到上一行
        df.loc[i-1, "nom de la revue"] = str(df.loc[i-1, "nom de la revue"]) + " " + str(df.loc[i, "nom de la revue"])
        # 也可以合并其他可能跨行的列
        # 删除当前行
        df.loc[i, "nom de la revue"] = None  # 可选，在错行上显示None
        df.loc[i, 'remove'] = True#并新增一列remove标记为True

# 删除标记行
df = df[df.get('remove') != True].reset_index(drop=True)
df = df.drop(columns=['remove'], errors='ignore')

## 
desired_order = ['nom de la revue', 'issn', 'rang', 'domaine']
df2022 = df[desired_order]
print(len(df2022))
display(df2022)

In [ ]:
# exception de l'exception:

import camelot
import pandas as pd

input_pdf = "fnege_pdf\FNEGE-classement-revues-de-gestion-2022.pdf"
# 坐标顺序： [x1, y1, x2, y2] （单位是 pt，PDF 坐标系左下原点）
# x1,y1 → 左上角; x2,y2 → 右下角
table_areas = ["30,750,650,100"]  

tables = camelot.read_pdf(
    input_pdf,
    pages="14",#all
    flavor="stream",  # 对复杂表格更稳健
    table_areas=table_areas
)
print("Tables détectées :", len(tables))

if len(tables)>0:
    df_list = [t.df for t in tables]
    df = pd.concat(df_list, ignore_index=True)


df.columns = ["issn", "nom de la revue", "domaine", "rang19", "rang"]

for i in range(1, len(df)):
    if pd.isna(df.loc[i, 'issn']) or df.loc[i, 'issn'] == "":
        # 将 title 合并到上一行
        df.loc[i-1, "nom de la revue"] = str(df.loc[i-1, "nom de la revue"]) + " " + str(df.loc[i, "nom de la revue"])
        # 也可以合并其他可能跨行的列
        # 删除当前行
        df.loc[i, "nom de la revue"] = None  # 可选，在错行上显示None
        df.loc[i, 'remove'] = True#并新增一列remove标记为True

# 删除标记行
df = df[df.get('remove') != True].reset_index(drop=True)
df = df.drop(columns=['remove'], errors='ignore')

import re
# df.iloc[1]['nom de la revue'].split('\n')[1]
#\s+ 匹配所有空白（包括空格、制表符、换行）;连字符“-”（转义以避免歧义）
df['nom de la revue']=df['nom de la revue'].apply(lambda x: re.sub(r"[\d*\-\n]","", x).strip() if isinstance(x, str) else x)

# 
desired_order = ['nom de la revue', 'issn', 'rang', 'domaine']
df_last = df[desired_order]
print(len(df_last))
display(df_last)


# test:

## concaténer tout :

In [5]:
import os
import pandas as pd
[f for f in os.listdir('fnege_tables') if f.endswith('.csv')]

['fnege_2011.csv',
 'fnege_2013.csv',
 'fnege_2016.csv',
 'fnege_2019.csv',
 'fnege_2022.csv',
 'fnege_final.csv']

In [5]:
import re
def clean_ponc(s):  
    s=s.strip().lower()
    clean_s=re.sub(r"[^\w\s]","", s)
    # \w\s → 匹配所有字母、数字、下划线、空白的字符（即标点符号）。
    # ^ 表示取反
    return clean_s

def read_clean_fnege(year):
    df = pd.read_csv(f"fnege_tables/fnege_{year}.csv")

    # 标准化列名
    df.columns = [c.strip().lower() for c in df.columns]

    # 统一命名
    rename_map = {
        "nom de la revue": "journal",
        "issn": "issn",
        "domaine": f"domaine_{year}",
        "rang": f"rang_{year}"
    }
    df = df.rename(columns=rename_map)

    # CHECK :仅保留需要的列（避免多余列影响 merge）
    keep = ["journal", "issn", f"domaine_{year}", f"rang_{year}"]
    df = df[[c for c in keep if c in df.columns]]

    ## 清洗列名：
    df['journal']=df['journal'].apply(clean_ponc)
    
    # 清洗domaine：
    df[f"domaine_{year}"] = (
        df[f"domaine_{year}"]
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "/", regex=True)
        .str.strip("/")
    )
    
    # 字母全部转为小写
    # 把所有非字母数字的连续符号（如空格、换行、斜杠、连字符）全部替换为一个 /
    # 删除开头与结尾的 /
    print(f"[INFO] fnege {year} : {len(df)} lines! {df.issn.nunique()} journaux!")
    display(df.head())
    return df

df11 = read_clean_fnege("2011")
df13 = read_clean_fnege("2013")
df16 = read_clean_fnege("2016")
df19 = read_clean_fnege("2019")
df22 = read_clean_fnege("2022")

[INFO] fnege 2011 : 401 lines! 400 journaux!


,journal,issn,domaine_2011,rang_2011
0,abacus,0001-3072,cpt,3
1,academy of management journal,0001-4273,mgtgen,1
2,academy of management review,0363-7425,mgtgen,1*
3,accounting and business research,0001-4788,cpt,3
4,accounting and finance,0810-5391,cpt,4


[INFO] fnege 2013 : 429 lines! 428 journaux!


,journal,issn,domaine_2013,rang_2013
0,grh,2034-9130,hrm,4
1,abacus,0001-3072,acc,2
2,academy of management annals,1941-6520,gen/man,3
3,academy of management journal,0001-4273,gen/man,1
4,academy of management \nperspectives,1558-9080,gen/man,2


[INFO] fnege 2016 : 464 lines! 463 journaux!


,journal,issn,domaine_2016,rang_2016
0,grh,2034-9130,hrm,3
1,abacus a journal of accounting \nfinance and b...,0001-3072,acc,2
2,academy of management annals,1941-6520,gen/man,1
3,academy of management journal,0001-4273,gen/man,1
4,academy of management learning \nand education,1537-260X,gen/man,2


[INFO] fnege 2019 : 484 lines! 484 journaux!


,journal,issn,domaine_2019,rang_2019
0,grh,2034-9130,hrm,3
1,abacus,0001-3072,acc,2
2,academy of management annals,1941-6520,gen/man,1
3,academy of management journal,0001-4273,gen/man,1
4,academy of management learning \nand education,1537-260X,gen/man,2


[INFO] fnege 2022 : 715 lines! 714 journaux!


,journal,issn,domaine_2022,rang_2022
0,grh,2034-9130,hrm,3
1,abacus,0001-3072,acc,2
2,academy of management annals,1941-6520,gen/man,1
3,academy of management discoveries,2168-1007,gen/man,4
4,academy of management journal,0001-4273,gen/man,1*


In [ ]:
def read_clean_fnege_v1(year):
    df = pd.read_csv(f"fnege_tables/fnege_{year}.csv")
    # 标准化列名
    df.columns = [c.strip().lower() for c in df.columns]

    # 统一命名
    rename_map = {
        "nom de la revue": "journal"
    #     "issn": "issn",
    #     "domaine": f"domaine_{year}",
    #     "rang": f"rang_{year}"
    }
    df = df.rename(columns=rename_map)

    # CHECK :仅保留需要的列（避免多余列影响 merge）
    keep = ["journal", "issn", f"domaine", f"rang"]
    df = df[[c for c in keep if c in df.columns]]    
    
    print(f"[INFO] fnege {year} : {len(df)} lines! {df.issn.nunique()} journaux!")
    print(f"[CHECK] {df['rang'].value_counts(dropna=False)}\n")
    # display(df.head())
    return df


years=['2011',"2013", "2016", "2019", "2022"]
dict_fnege=dict()
for y in years:
    df=read_clean_fnege_v1(y)
    dict_fnege[y]=df
print(dict_fnege.keys())
display(df.head())


[INFO] fnege 2011 : 401 lines! 400 journaux!
[CHECK] rang
3     157
4     114
2      81
1      41
1*      8
Name: count, dtype: int64

[INFO] fnege 2013 : 429 lines! 428 journaux!
[CHECK] rang
3     164
4     127
2      86
1      44
1*      8
Name: count, dtype: int64

[INFO] fnege 2016 : 464 lines! 463 journaux!
[CHECK] rang
3     172
4     132
2      99
1      52
1*      9
Name: count, dtype: int64

[INFO] fnege 2019 : 484 lines! 484 journaux!
[CHECK] rang
3     188
4     124
2     108
1      55
1*      9
Name: count, dtype: int64

[INFO] fnege 2022 : 715 lines! 714 journaux!
[CHECK] rang
4     253
3     213
2     143
1      72
1*     30
EM      4
Name: count, dtype: int64

dict_keys(['2011', '2013', '2016', '2019', '2022'])


,journal,issn,domaine,rang
0,'@GRH,2034-9130,HRM,3
1,ABACUS,0001-3072,ACC,2
2,ACADEMY OF MANAGEMENT ANNALS,1941-6520,GEN MAN,1
3,ACADEMY OF MANAGEMENT DISCOVERIES,2168-1007,GEN MAN,4
4,ACADEMY OF MANAGEMENT JOURNAL,0001-4273,GEN MAN,1*


In [ ]:
# extrait tous les id，set()

import re
def clean_issn(issn):
    if not isinstance(issn, str):
        return None
    # 去掉空格
    issn = issn.strip().replace(' ', '')
    # 如果已经是 XXXX-XXXX 格式，直接返回
    if re.match(r'^\d{4}-\d{3}[\dX]$', issn):
        return issn
    # 如果是连续 8 位数字，加上 '-'
    elif re.match(r'^\d{8}$', issn):
        return issn[:4] + '-' + issn[4:]
    else:
        # 不符合标准，返回原值或 None
        return issn


all_ids=[]
for y in years:
    df_y=dict_fnege[y]
    ids=list(df_y['issn'])
    
    ## clean issn: 出现这种情况：0959- 6526 !=0959-6526
    ids=[clean_issn(id) for id in ids]
    print(f"{y}: {len(ids)}")
    all_ids.extend(ids)

print(len(all_ids))
print(len(set(all_ids)))
# issn==id
# 782=>778


2011: 401
2013: 429
2016: 464
2019: 484
2022: 715
2493
778


In [ ]:
# cols : id, rang11~rang22, journal_names_set, journal_hal
# 搜索时候按照journal_hal返回rang11~rang22搜索
 
## 为每一个编号，历遍每一年找到一个journal name，可能不同年有差别! 最后set()
# 再在journal_name_hal中寻找对应名字,
# 在每一年内寻找rank
    
import pandas as pd
years = sorted(dict_fnege.keys())
rows = []

for issn_id in set(all_ids):
    row = {'issn': issn_id}
    
    journal_names = []
    
    for y in years:
        df_y = dict_fnege[y]
        row_match = df_y[df_y['issn'] == issn_id]
        
        if not row_match.empty:
            # 取该年的 rang 和 journal
            row[f'rang_{y}'] = row_match['rang'].iloc[0]  # 标量
            journal_names.append(row_match['journal'].iloc[0])
        else:
            row[f'rang_{y}'] = None  # 如果这一年没有该 ISSN
    
    # 去重保留顺序
    row['journal_names'] = '; '.join(list(dict.fromkeys(journal_names)))
    rows.append(row)
    # break # chq id

df_all = pd.DataFrame(rows)
print(len(df_all))
display(df_all.head())


778


,issn,rang_2011,rang_2013,rang_2016,rang_2019,rang_2022,journal_names
0,1962-2961,None,None,4,4,4,Revue Française de Gouvernance \nd'Entreprise;...
1,1740-8776,None,None,None,3,3,Management and Organization \nReview; MANAGEME...
2,1460-1060,4,4,4,4,4,European Journal of Innovation Management; Eur...
3,1090-6738,3,3,3,3,3,International Journal of Auditing; INTERNATION...
4,0957-1787,None,None,None,None,4,UTILITIES POLICY


In [ ]:
import json 
# import 
fnege_hal_path="fnege_data0\classement_fnege.json"
with open(fnege_hal_path, "r", encoding='utf-8') as f:
    fnege_hal=json.load(f)
print(len(fnege_hal))


##clean and match?
# PB 'journal_names'列中包含不同年份的写法（大小写、空格、特殊字符可能不同），而 fnege_hal 是 HAL 上的标准写法。
# 清洗journal names和journal hal，再进行匹
# 映射排名的时候现在journal hal上严格匹配，而不是再不干净的journal names上严格匹配

import re
def clean_journal_name(journal):
    journal = re.sub("\n","",journal.strip()) #.replace('\n', ' ')#去掉首位换行
    journal = re.sub(r'\s+', ' ', journal) # multi spaces=>one
    journal = re.sub(r'\s+', ' ', journal) # multi spaces=>one

    return journal.lower()

# find
def find_journals_hal(journals_str, fnege_hal):
    journal_list=journals_str.split(';')
    
    journal_list=[clean_journal_name(j) for j in journal_list]
    fnege_hal_keys_clean=[clean_journal_name(j) for j in fnege_hal.keys()]
    
    dict_hal=dict(zip(fnege_hal_keys_clean,fnege_hal.keys()))
    
    # 按照干净keys寻找，返回原始的hal keys
    journal_hal_list=set([dict_hal.get(j,None) for j in journal_list if j in fnege_hal_keys_clean])
    return '; '.join(journal_hal_list) if len(journal_hal_list)>0 else None
 

df_all['journal_hal'] = df_all['journal_names'].apply(lambda x : find_journals_hal(x, fnege_hal))
print(len(df_all))
display(df_all.head())

485
778


,issn,rang_2011,rang_2013,rang_2016,rang_2019,rang_2022,journal_names,journal_hal
0,1962-2961,None,None,4,4,4,Revue Française de Gouvernance \nd'Entreprise;...,Revue Française de Gouvernance d'Entreprise
1,1740-8776,None,None,None,3,3,Management and Organization \nReview; MANAGEME...,Management and Organization Review
2,1460-1060,4,4,4,4,4,European Journal of Innovation Management; Eur...,European Journal of Innovation Management
3,1090-6738,3,3,3,3,3,International Journal of Auditing; INTERNATION...,International Journal of Auditing
4,0957-1787,None,None,None,None,4,UTILITIES POLICY,None


In [62]:
df_all['journal_hal'].value_counts(dropna=False)

journal_hal
None                                                 288
Journal of Consumer Affairs                            2
Journal of Travel & Tourism Marketing                  2
Supply Chain Forum: An International Journal           2
Recherche et Cas en Sciences de Gestion                2
                                                    ... 
Politiques et Management Public                        1
La Revue des Sciences de Gestion                       1
International Journal of Public Sector Management      1
Medical Care                                           1
Finance Research Letters                               1
Name: count, Length: 474, dtype: int64

In [63]:
df_all[df_all['journal_hal']=="Journal of Consumer Affairs"]

,issn,rang_2011,rang_2013,rang_2016,rang_2019,rang_2022,journal_names,journal_hal
161,1745-6606,3,None,None,None,None,Journal of Consumer Affairs,Journal of Consumer Affairs
471,0022-0078,None,4,3,3,3,Journal of Consumer Affairs; JOURNAL OF CONSUM...,Journal of Consumer Affairs


In [ ]:
# no clean
# df_all.journal_hal.value_counts(dropna=False)

journal_hal
None                                              392
[Journal of Consumer Affairs]                       2
[Journal of Cleaner Production]                     2
[Contemporary Accounting Research]                  2
[Journal of Strategic Marketing]                    2
                                                 ... 
[European Journal of Public Health]                 1
[Journal of International Accounting Research]      1
[Financial Markets and Portfolio Management]        1
[International Labour Review]                       1
[Entrepreneurship Research Journal]                 1
Name: count, Length: 385, dtype: int64

## merge by journal hal!

In [ ]:
df_merged = df_all.groupby('journal_hal')
print(len(df_merged))
#有很多期刊还没有在hal上出现，但应该保留！！
#分开处理再concatenate


473


In [71]:
df_nonan = df_all[df_all['journal_hal'].notna()]
df_nan=df_all[~df_all['issn'].isin(df_nonan['issn'])]
# df_nan['journal_hal']=df_nan['journal_hal'].fillna()
print(len(df_nonan), len(df_nan), len(df_all)) # ✔
display(df_nan.head())

490 288 778


,issn,rang_2011,rang_2013,rang_2016,rang_2019,rang_2022,journal_names,journal_hal
4,0957-1787,None,None,None,None,4,UTILITIES POLICY,None
8,1389-5753,None,None,None,None,4,ELECTRONIC COMMERCE RESEARCH,None
9,0308-518X,None,None,None,None,2,ENVIRONMENT AND PLANNING A,None
12,1741-802X,None,None,None,None,4,INTERNATIONAL JOURNAL OF BUSINESS GOVERNANCE A...,None
18,0022-0388,None,None,None,None,3,JOURNAL OF DEVELOPMENT STUDIES,None


In [ ]:
import pandas as pd

df_merged = df_nonan.groupby('journal_hal').agg({
    'issn': 'first',  # 保留一个 ISSN（或者可以用 list 保存多个）
    'journal_names': lambda x: '; '.join(x),  # 合并 journal_names 列
    'rang_2011': 'first',  # 每年的排名列取第一个非空值
    'rang_2013': 'first',
    'rang_2016': 'first',
    'rang_2019': 'first',
    'rang_2022': 'first'
}).reset_index()
print(len(df_merged))
display(df_merged.head())


## concat : 
df_final=pd.concat([df_merged,df_nan],axis=0)
df_final=df_final[['issn','journal_names','journal_hal','rang_2011','rang_2013','rang_2016','rang_2019','rang_2022']]
print(len(df_final))
display(df_final.head())

df_final.to_csv('../external_data/fnege_final_hal.csv')

473


,journal_hal,issn,journal_names,rang_2011,rang_2013,rang_2016,rang_2019,rang_2022
0,@GRH,2034-9130,@grh; '@GRH,None,4,3,3,3
1,ACM Transactions on Computer-Human Interaction,1073-0516,ACM Transactions on Human Computer \nInteracti...,4,4,3,3,3
2,ASTIN Bulletin,0515-0361,ASTIN Bulletin: Journal of the International \...,3,3,3,3,3
3,Abacus,0001-3072,"Abacus; Abacus: A Journal of Accounting, \nFin...",3,2,2,2,2
4,Academy of Management Annals,1941-6520,Academy of Management Annals; ACADEMY OF MANAG...,None,3,1,1,1


761


,issn,journal_names,journal_hal,rang_2011,rang_2013,rang_2016,rang_2019,rang_2022
0,2034-9130,@grh; '@GRH,@GRH,None,4,3,3,3
1,1073-0516,ACM Transactions on Human Computer \nInteracti...,ACM Transactions on Computer-Human Interaction,4,4,3,3,3
2,0515-0361,ASTIN Bulletin: Journal of the International \...,ASTIN Bulletin,3,3,3,3,3
3,0001-3072,"Abacus; Abacus: A Journal of Accounting, \nFin...",Abacus,3,2,2,2,2
4,1941-6520,Academy of Management Annals; ACADEMY OF MANAG...,Academy of Management Annals,None,3,1,1,1


# appli : 2011-2022

In [6]:
import os, sys, importlib
sys.path.append(os.path.abspath(".."))

from utils import fnege
importlib.reload(fnege)
from utils.fnege import get_fnege_main

df_final=get_fnege_main(input_folder="../external_data/fnege_data/fnege_tables",
                        fnege_hal_path='../external_data/fnege_data/classement_fnege.json',
                        output_path="../external_data/fnege_final_hal2.csv")

----------------------------------------clean & reorder csv----------------------------------------- 

[INFO] fnege 2011 : 401 lines! 400 journaux!
[INFO] fnege 2013 : 429 lines! 428 journaux!
[INFO] fnege 2016 : 464 lines! 463 journaux!
[INFO] fnege 2019 : 484 lines! 484 journaux!
[INFO] fnege 2022 : 715 lines! 714 journaux!
[CHECK] years of fnege: dict_keys(['2011', '2013', '2016', '2019', '2022'])
--------------------------------------------get all issn-------------------------------------------- 

len 2011: 401
len 2013: 429
len 2016: 464
len 2019: 484
len 2022: 715
len(2011-2022) :778
---------------------concateante all df by issn: issn, rang_yrs, journal_names---------------------- 

[CHECK] len(df_all) 778
--------------------------find journal_names in journal by cleaning first--------------------------- 

----------------------------------merge df with journal_hal by it----------------------------------- 

[CHECK]490 df has corresponding journal hal; 288 doesnt
--------------